Step 1: Detect comment language

In [ ]:
import pandas as pd
from translation import detect_language

# Detection restricted to English / German (langdetect alone labelled short German
# comments as Hungarian, Catalan, Norwegian... and they were never translated)

# Load the cleaned comments
df = pd.read_csv("../data/cleaned_comments.csv")

# Create the new language column
df["comment_language"] = df["Comment"].apply(detect_language)

# Display the language distribution
print("Language distribution:")
print(df["comment_language"].value_counts())

Sanity Check

In [ ]:
# Display 5 German and 5 English comments to verify detection
print("\n=== Detected as GERMAN ===")
print(df[df["comment_language"] == "de"]["Comment"].head())
print("\n=== Detected as ENGLISH ===")
print(df[df["comment_language"] == "en"]["Comment"].head())

Step 2: Test Deepl API connection

In [ ]:
import deepl
from config import DEEPL_API_KEY

# Initialize the DeepL translator
translator = deepl.Translator(DEEPL_API_KEY)

# Test with a simple German sentence
result = translator.translate_text("Lieferverzögerung beim Lieferanten", target_lang="EN-US")
print("Translation test:", result.text)

# Check remaining character quota (Free plan)
usage = translator.get_usage()
print(f"\nCharacters used: {usage.character.count}")
print(f"Character limit: {usage.character.limit}")

Step 3: Translate DE comments to EN using DeepL (with the translation cache: each comment is paid for once)

In [ ]:
import pandas as pd
from translation import add_translations

df = pd.read_csv("../data/cleaned_comments.csv")

# dry_run=True only counts the characters that would be sent to DeepL
df = add_translations(df, dry_run=False)

df.to_csv("../data/cleaned_comments.csv", index=False)
print("\n✅ Translation complete! Saved to data/cleaned_comments.csv")

# --- Verify ---
print("\n=== Before / After examples ===")
sample = df[df["comment_language"] == "de"][["Comment", "Comment_EN"]].head(5)
for _, row in sample.iterrows():
    print(f"DE: {row['Comment']}")
    print(f"EN: {row['Comment_EN']}\n")